In [ ]:
import ztfssoquery
from pathlib import Path

In [ ]:
targetname = "24P"

WORKDIR  = Path.cwd() / ".."

outdir = WORKDIR / "data" / "".join(targetname.split())
outdir.mkdir(parents=True, exist_ok=True)
outdir

FIGDIR = outdir / "fig"
FIGDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Run the full query and URL generation process
ztfssoquery.generate_fits_urls(
    primary_des=targetname,
    start_date='2025-07-01',
    end_date='2025-12-31',
    interval_days=5,
    rh_max=9,
    vmag_max=20,
    is_cutout=True,
    cutout_size="10arcmin",
    output_dir=outdir
)

In [ ]:
ztfssoquery.import_fitsurl(outdir/"fits_urls.txt", output_dir=outdir)

## Check example data

In [ ]:
import ccdproc 
from astropy.io import fits

allfits = ccdproc.ImageFileCollection(outdir, glob_include="*.fits").filter()
summary = allfits.summary.to_pandas()
summary.head()

In [ ]:
fpath_fits = summary.iloc[-2]["file"]

hdul = fits.open(fpath_fits)
header = hdul[0].header
hdul.info()

In [ ]:
header

In [ ]:
from astropy.wcs import WCS

wcs = WCS(hdul[0].header)
wcs

In [ ]:
from astropy.time import Time

obstime = Time(hdul[0].header['OBSJD'], format='jd')

from astroquery.jplhorizons import Horizons
target_id = 90000355  # 24P/Schaumasse
obj = Horizons(id=target_id, location='I41', epochs=obstime.jd)
eph = obj.ephemerides().to_pandas().iloc[0]

# ztfssoquery.query_sso_ephemeris(targetname, obstime.jd, ephem_quantities="1,3,9,19")

## Figure example

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval
import _rcparams
import numpy as np

fig = plt.figure()
ax = fig.add_subplot(projection=wcs)
vmin, vmax = ZScaleInterval().get_limits(hdul[0].data)
ax.imshow(hdul[0].data, cmap='gray', vmin=vmin, vmax=vmax, origin='lower')

text_annotate = f"""OBSERVAT = {hdul[0].header['ORIGIN']}
DATE-OBS = {obstime.isot}
FILTER   = {hdul[0].header['FILTER']}
EXPTIME  = {hdul[0].header['EXPTIME']} sec

OBJECT   = {targetname}
Rh       = {eph['r']:.3f} AU
Delta    = {eph['delta']:.3f} AU
Phase    = {eph['alpha']:.1f} deg
S-T PA   = {eph.sunTargetPA:.1f} deg
Vel PA   = {eph.velocityPA:.1f} deg
"""

ax.annotate(text_annotate, xy=(1.05, 0.95), xycoords='axes fraction',
            fontsize=15, color='k', 
            va="top", ha="left", family="monospace")

arrow_style = dict(facecolor="white", edgecolor="white", width=3, head_width=10, head_length=5)
arrow_text_style = dict(color="white", ha="center", va="center", fontsize=15, weight="bold")
ny, nx = hdul[0].data.shape
base_x, base_y = 0.2 * nx, 0.8 * ny
arrow_length = 0.13 * min(nx, ny)

# North Arrow
ax.arrow(base_x, base_y,
         0, -arrow_length, 
         **arrow_style)
ax.text(base_x, base_y - arrow_length * 1.4,
        'N', **arrow_text_style)

# East Arrow
ax.arrow(base_x, base_y,
         -arrow_length, 0,
         **arrow_style)
ax.text(base_x - arrow_length * 1.4,
        base_y,
        'E', **arrow_text_style)

# velocity PA Arrow
vel_pa_rad = np.deg2rad(eph.velocityPA)
ax.arrow(base_x, base_y,
         -arrow_length * np.sin(vel_pa_rad),
         -arrow_length * np.cos(vel_pa_rad),
         **arrow_style)
ax.text(base_x - arrow_length * 1.4 * np.sin(vel_pa_rad),
        base_y - arrow_length * 1.4 * np.cos(vel_pa_rad),
        '$-V$', **arrow_text_style)

# sun Target PA Arrow
sun_pa_rad = np.deg2rad(eph.sunTargetPA)
ax.arrow(base_x, base_y,
         -arrow_length * np.sin(sun_pa_rad),
         -arrow_length * np.cos(sun_pa_rad),
         **arrow_style)
ax.text(base_x - arrow_length * 1.4 * np.sin(sun_pa_rad),
        base_y - arrow_length * 1.4 * np.cos(sun_pa_rad),
        '$-\u2609$', **arrow_text_style) 

plt.savefig(FIGDIR/f"{header['FILENAME']}.png")
plt.show()

In [ ]:
###
### All figure to save
###

for idx, row in summary.iterrows():
    
    fpath_fits = row["file"]
    hdul = fits.open(fpath_fits)
    header = hdul[0].header
    wcs = WCS(hdul[0].header)
    obstime = Time(hdul[0].header['OBSJD'], format='jd')
    obj = Horizons(id=target_id, location='I41', epochs=obstime.jd)
    eph = obj.ephemerides().to_pandas().iloc[0]

    fig = plt.figure()
    ax = fig.add_subplot(projection=wcs)
    vmin, vmax = ZScaleInterval().get_limits(hdul[0].data)
    ax.imshow(hdul[0].data, cmap='gray', vmin=vmin, vmax=vmax, origin='lower')

    text_annotate = f"""OBSERVAT = {hdul[0].header['ORIGIN']}
DATE-OBS = {obstime.isot}
FILTER   = {hdul[0].header['FILTER']}
EXPTIME  = {hdul[0].header['EXPTIME']} sec
OBJECT   = {targetname}
Rh       = {eph['r']:.3f} AU
Delta    = {eph['delta']:.3f} AU
Phase    = {eph['alpha']:.1f} deg
S-T PA   = {eph.sunTargetPA:.1f} deg
Vel PA   = {eph.velocityPA:.1f} deg
"""

    ax.annotate(text_annotate, xy=(1.05, 0.95), xycoords='axes fraction',
                fontsize=15, color='k', 
                va="top", ha="left", family="monospace")
    
    arrow_style = dict(facecolor="white", edgecolor="white", width=3, head_width=10, head_length=5)
    arrow_text_style = dict(color="white", ha="center", va="center", fontsize=15, weight="bold")
    ny, nx = hdul[0].data.shape
    base_x, base_y = 0.2 * nx, 0.8 * ny
    arrow_length = 0.13 * min(nx, ny)
    # North Arrow
    ax.arrow(base_x, base_y,
             0, -arrow_length, 
             **arrow_style)
    ax.text(base_x, base_y - arrow_length * 1.4,
            'N', **arrow_text_style)  
    
    # East Arrow
    ax.arrow(base_x, base_y,
             -arrow_length, 0,
             **arrow_style)
    ax.text(base_x - arrow_length * 1.4,
            base_y,
            'E', **arrow_text_style)
    # velocity PA Arrow
    vel_pa_rad = np.deg2rad(eph.velocityPA)
    ax.arrow(base_x, base_y,
                -arrow_length * np.sin(vel_pa_rad),
                -arrow_length * np.cos(vel_pa_rad),
                **arrow_style)
    ax.text(base_x - arrow_length * 1.4 * np.sin(vel_pa_rad),
            base_y - arrow_length * 1.4 * np.cos(vel_pa_rad),
            '$-V$', **arrow_text_style)
    # sun Target PA Arrow
    sun_pa_rad = np.deg2rad(eph.sunTargetPA)
    ax.arrow(base_x, base_y,
                -arrow_length * np.sin(sun_pa_rad),
                -arrow_length * np.cos(sun_pa_rad),
                **arrow_style)
    ax.text(base_x - arrow_length * 1.4 * np.sin(sun_pa_rad),
            base_y - arrow_length * 1.4 * np.cos(sun_pa_rad),
            '$-\u2609$', **arrow_text_style)    
    
    plt.savefig(FIGDIR/f"{header['FILENAME']}.png")
    plt.close()  